Now, use the newly targeted Bronze volume path. This code reads active location keys from your Gold master, calls AccuWeather, and drops the raw json files precisely inside your stweatherprojectnew container.

In [0]:
# %sql
# -- 1. Grant Databricks permission to touch the Bronze Container
# CREATE EXTERNAL LOCATION IF NOT EXISTS bronze_storage_location
# URL 'abfss://bronze@stweatherprojectnew.dfs.core.windows.net/'
# WITH (CREDENTIAL weather_credential);

# -- 2. Ensure your metadata schema exists
# CREATE SCHEMA IF NOT EXISTS weather_catalog.bronze;

# -- 3. Create an EXTERNAL Volume mapping explicitly to your Bronze container
# CREATE EXTERNAL VOLUME IF NOT EXISTS weather_catalog.bronze.raw_forecasts
# LOCATION 'abfss://bronze@stweatherprojectnew.dfs.core.windows.net/forecasts/raw/';

In [0]:
# Databricks notebook source
import requests
import json
import os

# 1. API Configurations
API_KEY = ""
BASE_URL = "https://dataservice.accuweather.com/forecasts/v1/daily/5day/"

# This POSIX path directly connects to your Bronze container raw directory
bronze_volume_path = "/Volumes/weather_catalog/bronze/raw_forecasts/"

# 2. Extract keys from your Location Master
location_keys_df = spark.table("weather_catalog.gold.location_master").select("key").distinct()
location_keys = [row["key"] for row in location_keys_df.collect()]

print(f"Found {len(location_keys)} locations to fetch forecasts for.")

# 3. Pull from API and save raw data directly to Bronze
for key in location_keys:
    try:
        request_url = f"{BASE_URL}{key}?apikey={API_KEY}&details=true&metric=true"
        response = requests.get(request_url)
        
        if response.status_code == 200:
            raw_data = response.json()
            raw_data["LocationKey"] = key # Metadata tracking
            
            # Create subfolder path per location inside Bronze Volume
            target_dir = f"{bronze_volume_path}location_{key}"
            dbutils.fs.mkdirs(target_dir)
            
            # Write raw JSON directly to the cloud storage bucket
            dbutils.fs.put(f"{target_dir}/forecast.json", json.dumps(raw_data), overwrite=True)
            print(f"Successfully saved raw forecast for Location Key: {key} into Bronze.")
        else:
            print(f"Failed to fetch data for key {key}. Status code: {response.status_code}")
            
    except Exception as e:
        print(f"Error processing location key {key}: {str(e)}")

Found 3 locations to fetch forecasts for.
Wrote 25242 bytes.
Successfully saved raw forecast for Location Key: 204842 into Bronze.
Wrote 25598 bytes.
Successfully saved raw forecast for Location Key: 202396 into Bronze.
Wrote 24923 bytes.
Successfully saved raw forecast for Location Key: 204848 into Bronze.
